# Step 01 — Gemma 4 E2B Baseline + Function Calling

**Phase:** 1 — Model Access & Function Calling  
**Goal:** Load Gemma 4 E2B, verify generation, and implement function calling schema for the Emotion Radar API.  
**Environment:** Kaggle / Colab (GPU T4 or better)  

---

**Expected output:**  
A JSON object from a function call — emotion radar result for a sample transcript snippet.

In [ ]:
# Cell 2 — Environment setup
import subprocess, sys

def install(pkg):
    subprocess.check_call([sys.executable, '-m', 'pip', 'install', pkg, '-q'])

install('transformers>=4.40.0')
install('accelerate>=0.27.0')
install('bitsandbytes>=0.43.0')

# Detect environment
try:
    import google.colab
    ENV = 'colab'
except ImportError:
    try:
        import kaggle_secrets
        ENV = 'kaggle'
    except ImportError:
        ENV = 'local'

print(f'Environment: {ENV}')

In [ ]:
# Cell 3 — Imports
import json
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig

In [ ]:
# Cell 4 — Config (all tunable values in one place)
MODEL_ID = 'google/gemma-3-2b-it'   # Swap to gemma-4 when available on Hub
LOAD_IN_4BIT = True                  # Set False if you have A100+
MAX_NEW_TOKENS = 512
DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'

print(f'Device: {DEVICE}')
print(f'Model: {MODEL_ID}')
print(f'4-bit: {LOAD_IN_4BIT}')

In [ ]:
# Cell 5 — Load model
bnb_config = BitsAndBytesConfig(
    load_in_4bit=LOAD_IN_4BIT,
    bnb_4bit_use_double_quant=True,
    bnb_4bit_quant_type='nf4',
    bnb_4bit_compute_dtype=torch.bfloat16,
) if LOAD_IN_4BIT else None

tokenizer = AutoTokenizer.from_pretrained(MODEL_ID)
model = AutoModelForCausalLM.from_pretrained(
    MODEL_ID,
    quantization_config=bnb_config,
    device_map='auto',
    torch_dtype=torch.bfloat16,
)

print('Model loaded.')
print(f'Parameters: {model.num_parameters() / 1e9:.2f}B')

In [ ]:
# Cell 6 — Function calling schema: Emotion Radar
EMOTION_RADAR_SCHEMA = {
    'name': 'analyze_emotion',
    'description': 'Analyze the emotional state of a conversation turn. Returns structured emotion data for both speakers.',
    'parameters': {
        'type': 'object',
        'properties': {
            'speaker': {
                'type': 'string',
                'description': 'Which speaker: user or other'
            },
            'turn_text': {
                'type': 'string',
                'description': 'The utterance to analyze'
            },
            'emotions': {
                'type': 'object',
                'properties': {
                    'primary': {'type': 'string', 'enum': ['anger', 'fear', 'sadness', 'joy', 'surprise', 'disgust', 'neutral', 'frustration', 'openness']},
                    'intensity': {'type': 'number', 'minimum': 0.0, 'maximum': 1.0},
                    'tension_level': {'type': 'number', 'minimum': 0.0, 'maximum': 1.0},
                    'defensive': {'type': 'boolean'},
                    'concession_made': {'type': 'boolean'},
                }
            },
            'whisper_prompt': {
                'type': 'string',
                'description': 'Optional real-time coaching tip for the user. Null if no action needed.'
            }
        },
        'required': ['speaker', 'turn_text', 'emotions']
    }
}

print('Schema defined:')
print(json.dumps(EMOTION_RADAR_SCHEMA, indent=2))

In [ ]:
# Cell 7 — Prompt engineering for function calling
SYSTEM_PROMPT = """You are Tough Talks' conversation intelligence engine.
Your job is to analyze conversation turns and return structured JSON output — no prose, no explanation.
Always respond with a valid JSON object matching the requested schema.
Be precise about emotion detection. Do not over-amplify — neutral is a valid state."""

SAMPLE_TURN = "Look, I've been here 3 years and I think my work speaks for itself. \
I'm not trying to be difficult, I just... I feel like the number we discussed is fair."

USER_PROMPT = f"""Analyze this conversation turn using the analyze_emotion function.
Speaker: user
Turn: \"{SAMPLE_TURN}\"

Return ONLY a JSON object matching this schema:
{json.dumps(EMOTION_RADAR_SCHEMA['parameters']['properties'], indent=2)}"""

print('Prompts ready.')

In [ ]:
# Cell 8 — Run inference
chat = [
    {'role': 'system', 'content': SYSTEM_PROMPT},
    {'role': 'user', 'content': USER_PROMPT}
]

input_ids = tokenizer.apply_chat_template(
    chat, tokenize=True, add_generation_prompt=True, return_tensors='pt'
).to(DEVICE)

with torch.inference_mode():
    output = model.generate(
        input_ids,
        max_new_tokens=MAX_NEW_TOKENS,
        do_sample=False,          # Greedy for structured output
        temperature=None,
        top_p=None,
    )

response_ids = output[0][input_ids.shape[-1]:]
raw_response = tokenizer.decode(response_ids, skip_special_tokens=True)
print('Raw response:')
print(raw_response)

In [ ]:
# Cell 9 — Parse and validate output
import re

def extract_json(text):
    """Extract JSON from model output, handling markdown fences."""
    # Strip markdown fences
    text = re.sub(r'```json\s*', '', text)
    text = re.sub(r'```\s*', '', text)
    text = text.strip()
    return json.loads(text)

try:
    result = extract_json(raw_response)
    print('\n✅ Valid JSON output:')
    print(json.dumps(result, indent=2))
    
    # Basic validation
    assert 'emotions' in result, 'Missing emotions key'
    assert 'intensity' in result['emotions'], 'Missing intensity'
    print('\n✅ Schema validation passed')
    
except (json.JSONDecodeError, AssertionError) as e:
    print(f'\n❌ Validation failed: {e}')
    print('Raw output was:', raw_response)
    print('\n→ Action needed: adjust prompt or parsing in Cell 7/9')

## Step 01 Checklist

Before marking this step DONE:

- [ ] Model loads without OOM error
- [ ] Inference runs in < 10s on T4 GPU
- [ ] JSON output parses cleanly
- [ ] `emotions.intensity` is a float 0–1
- [ ] `whisper_prompt` is present (or null)
- [ ] Observations added to `/knowledge/phases/`
- [ ] `PROGRESS.md` updated

**Report back with:**
1. The raw response (or any error)
2. GPU memory used (`nvidia-smi`)
3. Inference latency
4. Any prompt changes you made